In [0]:
# Initialize widgets. Use empty strings for paths to keep the code reusable,
# but default the scope to "default2" since that is verified shared vault
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "")
dbutils.widgets.text("secret_scope", "default2")

# Consolidate parameters into a structured configuration state
CONFIG = {
    "storage_account": dbutils.widgets.get("storage_account").strip(),
    "container": dbutils.widgets.get("container").strip(),
    "secret_scope": dbutils.widgets.get("secret_scope").strip(),
    "mount_point": f"/mnt/{dbutils.widgets.get('container').strip()}"
}

# Safely isolate lookup secrets
SECRET_KEYS = {
    "client_id": "sp-databricks-adls-appid",
    "client_secret": "sp-databricks-adls-appkey",
    "tenant_id": "tenant-id"
}

### 2. Service Principal OAuth Mounting Routine
This function acts defensively. It checks if the mount path is already active before trying to provision it, preventing runtime errors on repeated job executions.

In [0]:
def execute_legacy_mount(cfg: dict, keys: dict):
    target_source_uri = f"abfss://{cfg['container']}@{cfg['storage_account']}.dfs.core.windows.net/"
    
    try:
        active_mounts = dbutils.fs.mounts()
        if any(m.mountPoint == cfg["mount_point"] for m in active_mounts):
            print(f"Mount point already active at: {cfg['mount_point']}.")
            return True
    except Exception as e:
        if "DbfsDisabledException" in str(e) or "disabled" in str(e).lower():
            print("\n⚠️ LEGACY SECURITY PROTECTION ON: DBFS mounts are locked down.")
            return False

    try:
        print(f"Resolving credentials from scope: {cfg['secret_scope']}")
        
        # Safely pull the passwords out of your Key Vault scope
        sp_id = dbutils.secrets.get(scope=cfg['secret_scope'], key=keys['client_id'])
        sp_secret = dbutils.secrets.get(scope=cfg['secret_scope'], key=keys['client_secret'])
        tenant_id = dbutils.secrets.get(scope=cfg['secret_scope'], key=keys['tenant_id'])
        
        # Keep the ugly Hadoop configuration strings hidden in a tidy layout
        hadoop_settings = {
            "fs.azure.account.auth.type": "OAuth",
            "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            "fs.azure.account.oauth2.client.id": sp_id,
            "fs.azure.account.oauth2.client.secret": sp_secret,
            "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }
        
        print(f"Mapping mount point to: {cfg['mount_point']}")
        dbutils.fs.mount(source=target_source_uri, mount_point=cfg["mount_point"], extra_configs=hadoop_settings)
        return True

    except Exception as error:
        if "DbfsDisabledException" in str(error) or "disabled" in str(error).lower():
            print("\n⚠️ LEGACY SECURITY PROTECTION ON: Workspace cluster policy blocked the old-school mount path.")
        else:
            print(f"Mount failed: {error}")
        return False

# Execution trigger
MOUNT_SUCCESSFUL = False
if CONFIG["storage_account"] and CONFIG["container"]:
    MOUNT_SUCCESSFUL = execute_legacy_mount(CONFIG, SECRET_KEYS)
else:
    print("Execution halted: Fill out the widgets at the top of your screen first.")

In [0]:
# Safe review check - completely bypasses the call if widgets are empty or if security blocked the mount
if CONFIG["storage_account"] and CONFIG["container"] and MOUNT_SUCCESSFUL:
    try:
        display(dbutils.fs.ls(CONFIG["mount_point"]))
    except Exception as check_error:
        # Strip away the multi-line JVM trace and only print the top line text
        clean_error = str(check_error).splitlines()[0]
        print(f"Could not scan directory: {clean_error}")
else:
    print("Directory scan safely bypassed: Mount was not created or public DBFS root is disabled.")